<a href="https://colab.research.google.com/github/RamcharanChandragiri/NATURAL-LANGUAGE-PROCESSING/blob/main/NLP_ASSIGNMENT_16_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sequence Modeling II: Named Entity Recognition (NER)
using spaCy and Hugging Face

Section 3: Sequence Modeling – Named Entity Recognition

Aim:
To implement Named Entity Recognition (NER) using both spaCy
and Hugging Face Transformers, and compare their performance on
real-world text data.

In [29]:
# Install packages (Run once in Colab)
!pip install spacy transformers datasets evaluate seqeval -q
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
import spacy
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)
from seqeval.metrics import classification_report, f1_score

In [31]:
dataset = load_dataset("lhoestq/conll2003")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [32]:
#display the sample data
print(dataset["train"][0])

#label names
label_names = [
    'O',
    'B-PER', 'I-PER',
    'B-ORG', 'I-ORG',
    'B-LOC', 'I-LOC',
    'B-MISC', 'I-MISC'
]

print(label_names)



{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [33]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names)
)

print("Tokenizer and model loaded successfully")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Tokenizer and model loaded successfully


In [34]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Function created successfully")

Function created successfully


In [35]:
#preprocessing the dataset
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print("Dataset tokenized successfully")

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Dataset tokenized successfully


In [36]:
#creating the training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01
)

print("Training arguments set successfully")

Training arguments set successfully


In [37]:
#data collector
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

print("Data collator created successfully")

Data collator created successfully


In [38]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

print("Trainer created successfully")

Trainer created successfully


In [39]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.243367,0.071692
2,0.048910,0.062753


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.11056878300625533, metrics={'train_runtime': 382.4511, 'train_samples_per_second': 73.426, 'train_steps_per_second': 4.591, 'total_flos': 701093358405576.0, 'train_loss': 0.11056878300625533, 'epoch': 2.0})

In [40]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.06275348365306854, 'eval_runtime': 10.4312, 'eval_samples_per_second': 311.566, 'eval_steps_per_second': 19.557, 'epoch': 2.0}


In [41]:
import numpy as np
from seqeval.metrics import classification_report, f1_score

# Predict on test set
predictions, labels, _ = trainer.predict(tokenized_dataset["test"])

# Convert logits to predicted class ids
predictions = np.argmax(predictions, axis=2)

true_labels = []
pred_labels = []

# Convert numeric labels to tag names
for pred, label in zip(predictions, labels):
    true = []
    pred_out = []

    for p, l in zip(pred, label):
        if l != -100:
            true.append(label_names[l])
            pred_out.append(label_names[p])

    true_labels.append(true)
    pred_labels.append(pred_out)

# Print final metrics
print("F1 Score:", f1_score(true_labels, pred_labels))
print("\nClassification Report:\n")
print(classification_report(true_labels, pred_labels))

F1 Score: 0.8909935175394235

Classification Report:

              precision    recall  f1-score   support

         LOC       0.90      0.92      0.91      3000
        MISC       0.72      0.72      0.72      1266
         ORG       0.88      0.91      0.89      3524
         PER       0.94      0.94      0.94      2989

   micro avg       0.88      0.90      0.89     10779
   macro avg       0.86      0.87      0.87     10779
weighted avg       0.88      0.90      0.89     10779



In [42]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = "Virat Kohli played in Chennai on Monday."

doc = nlp(text)

print("Entities extracted using spaCy:")
for ent in doc.ents:
    print(ent.text, "-->", ent.label_)

Entities extracted using spaCy:
Virat Kohli --> PERSON
Chennai --> GPE
Monday --> DATE
